In [1]:
try:
    # faster implementation using bindings to libxml
    from lxml import etree as ET
except ImportError:
    print 'Falling back to default ElementTree implementation'
    from xml.etree import ElementTree as ET

from itertools import combinations
import json, csv, Levenshtein, sys

def get_elements(fp, tag):
    '''
        Convenience and memory management function
        that iterates required tags
    '''
    context = iter(ET.iterparse(fp, events=('start', 'end')))
    _, root = next(context)  # get root element
    for event, elem in context:
        if event == 'end' and elem.tag == tag:
            yield elem
            root.clear()  # preserve memory

def query_yes_no(question, default="yes"):
    """Ask a yes/no question via raw_input() and return their answer.

    "question" is a string that is presented to the user.
    "default" is the presumed answer if the user just hits <Enter>.
        It must be "yes" (the default), "no" or None (meaning
        an answer is required of the user).

    The "answer" return value is True for "yes" or False for "no".
    """
    valid = {"yes": True, "y": True, "ye": True,
             "no": False, "n": False}
    if default is None:
        prompt = " [y/n] "
    elif default == "yes":
        prompt = " [Y/n] "
    elif default == "no":
        prompt = " [y/N] "
    else:
        raise ValueError("invalid default answer: '%s'" % default)

    while True:
        sys.stdout.write(question + prompt)
        choice = raw_input().lower()
        if default is not None and choice == '':
            return valid[default]
        elif choice in valid:
            return valid[choice]
        else:
            sys.stdout.write("Please respond with 'yes' or 'no' "
                             "(or 'y' or 'n').\n")

name_map = {}
not_name_map = {}

Falling back to default ElementTree implementation


In [4]:
import csv, json

xml = '/home/yizhao/Downloads/pubmed_result.xml'
websites = '/home/yizhao/Downloads/pi_info.csv'
pis_old = '/home/yizhao/Downloads/authors_old.json'

articles = []
pis = []

with open(pis_old) as jsonfile:
    pi_map = {}
    for p in json.load(jsonfile):
        pi_map[p['name']] = p

with open(websites) as csvfile:
    csvreader = csv.reader(csvfile, delimiter=',', quotechar='"')
    maxId = 0
    people = []
    for row in csvreader:
        if row[0]:
            if row[0] in pi_map:
                pis.append({
                        'name': row[0],
                        'url': row[1],
                        'id': maxId,
                        'color': pi_map[row[0]]['color'],
                        'three': pi_map[row[0]]['three']
                    })
            else:
                pis.append({
                        'name': row[0],
                        'url': row[1],
                        'id': maxId,
                    })
            people.append(row[0])
            maxId = maxId + 1
    
for e in get_elements(xml, 'PubmedArticle'):
    year = e.findtext('MedlineCitation/DateCreated/Year')
    month = e.findtext('MedlineCitation/DateCreated/Month')
    pmid = e.findtext('MedlineCitation/PMID')
    article_title = e.findtext('MedlineCitation/Article/ArticleTitle')
    journal_title = e.findtext('MedlineCitation/Article/Journal/Title')
    
    collaborator = []
    authors = []
    
    for author in e.findall('MedlineCitation/Article/AuthorList/Author'):
        name = unicode(author.findtext('ForeName')) + ' ' + unicode(author.findtext('LastName'))
        pi_name = None
            
        if name.strip(' ') in name_map:
            name = unicode(name_map[name.strip(' ')])
            
        for pi in people:
            ratio = Levenshtein.ratio(unicode(pi), name)
            if ratio >= 0.9:
                pi_name = pi
            elif ratio < 0.9 and ratio >= 0.7:
                if name in not_name_map:
                    if not_name_map[name.strip(' ')] == pi:
                        break
                if query_yes_no(name + '\n' + pi + '\n'):
                    pi_name = pi
                    name_map[name.strip(' ')] = pi
                else:
                    not_name_map[name.strip(' ')] = pi

        if pi_name is not None:
            collaborator.append(pi_name)

    if len(collaborator) >= 1:
        collaborator_comb = []
        collaborator_comb.extend(combinations(collaborator, 2))
        
        for c in collaborator_comb:
            articles.append({
                    'pmid': pmid,
                    's': people.index(c[0]),
                    't': people.index(c[1]),
                    'jt': journal_title,
                    'at': article_title,
                    'd': '%d%02d'%(int(year), int(month)),
                })

with open('/home/yizhao/Downloads/articles.json', 'w') as outfile:
    json.dump(articles, outfile)
    
with open('/home/yizhao/Downloads/authors.json', 'w') as outfile:
    json.dump(pis, outfile)

print 'DONE'

Charles Yoon
Charles Boone
 [Y/n] n
Charles Vinson
Charles Boone
 [Y/n] n
Charles Brenner
Charles Boone
 [Y/n] n
DONE


In [ ]:
print name_map
print not_name_map